# 介紹

本程式使用 pyautogui 來完成 Biltz 的遊戲機器人

In [6]:
import pyautogui as pyg
pyg.sleep(2)
basex=1444
basey=398
gap=45


In [7]:
def get_gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心的 (x, y) 座標（列、欄從 0 開始）。"""
    return basex + c * gap, basey + r * gap

In [ ]:
def get_gem_color(r, c, im):
    """回傳截圖 im 中第 r 列、第 c 欄寶石中心的顏色。"""
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))

In [ ]:
def _can_move(r, c, dr, dc, im):
    """判斷將 (r, c) 與指定方向相鄰寶石交換後是否形成三連。"""
    nr, nc = r + dr, c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    original_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return original_color
        return get_gem_color(row, col, im)

    # 三連線必須包含交換後的其中一格。
    for row, col in ((r, c), (nr, nc)):
        for step_r, step_c in ((0, 1), (1, 0)):
            for offset in (-2, -1, 0):
                cells = [(row + (offset + i) * step_r,
                          col + (offset + i) * step_c) for i in range(3)]
                if all(0 <= rr < 8 and 0 <= cc < 8 for rr, cc in cells):
                    colors = [color_after_swap(rr, cc) for rr, cc in cells]
                    if colors[0] == colors[1] == colors[2]:
                        return True
    return False

def can_move_left(r, c, im):
    return _can_move(r, c, 0, -1, im)

def can_move_right(r, c, im):
    return _can_move(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_move(r, c, -1, 0, im)

def can_move_down(r, c, im):
    return _can_move(r, c, 1, 0, im)

In [8]:
print(get_gem_position(0,0))
print(get_gem_position(0,1))
print(get_gem_position(1,0))


(1444, 398)
(1489, 398)
(1444, 443)


In [ ]:
for i in range(8):
    x,y=get_gem_position(0,i)
    pyg.click(x,y,duration=1)
    print(get_gem_position(0,i,im))


In [ ]:
import pyautogui

# 輸入左上角寶石中心座標，以及相鄰寶石中心的間距
basex = int(input("左上角寶石中心的 x 座標 basex："))
basey = int(input("左上角寶石中心的 y 座標 basey："))
gap = int(input("相鄰寶石中心的間距 gap："))

# 擷取目前畫面，讀取 8×8 個寶石中心點的 RGB 顏色
screen = pyautogui.screenshot()

for row in range(8):
    colors = []
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        colors.append(screen.getpixel((x, y)))
    print(f"第 {row + 1} 行：{colors}")

In [ ]:
import time

start_time = time.time()
im = pyg.screenshot()
directions = (
    (can_move_left, 0, -1),
    (can_move_right, 0, 1),
    (can_move_up, -1, 0),
    (can_move_down, 1, 0),
)

while time.time() - start_time < 60:
    moved = False
    for r in range(8):
        for c in range(8):
            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    x1, y1 = get_gem_position(r, c)
                    x2, y2 = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(x1, y1)
                    pyg.dragTo(x2, y2, duration=0.15, button="left")
                    pyg.sleep(0.15)
                    im = pyg.screenshot()
                    moved = True
                    break
            if moved:
                break
        if moved:
            break
    if not moved:
        pyg.sleep(0.05)
        im = pyg.screenshot()